# Week 6 Student Worksheet: Spatial Prediction Shootout

## Traditional Stats vs. Machine Learning: Filling the Gaps

> *"Kriging tells you where it's uncertain. ML just says 'trust me.'"*

### Today's Mission

Transform Week 5's discrete rainfall stations into **continuous rainfall surfaces** using **two fundamentally different approaches**:

1. **Kriging (Statistical)** — uses spatial correlation + provides uncertainty
2. **Random Forest (ML)** — uses data patterns + easily adds features

Then compare them head-to-head with two simpler methods (Nearest Neighbor, IDW) and determine which gives the Commander more actionable intelligence.

> Fill in the code cells marked with `# YOUR CODE HERE`. Use AI tools strategically — but understand every line you write.

In [ ]:
# Install required packages (run once)
%pip install pykrige scikit-learn rasterio rasterstats --quiet

---

## Cell [1]: Environment Setup & Data Loading (Slide 2)

Load the Fung-wong typhoon data and filter to the study area. Reuse your Week 5 `parse_rainfall_json()` function.

**AI Prompt Suggestion**:
```
I need to load fungwong_202511.json and parse it into a GeoDataFrame using
my Week 5 parse_rainfall_json() function. Then filter to 花蓮縣 and 宜蘭縣,
remove -998/zero rain stations, and convert to EPSG:3826.
Show me the top 5 stations by rain_1hr.
```

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from shapely.geometry import Point
warnings.filterwarnings('ignore')

# YOUR CODE HERE:
# 1. Define normalize_cwa_json() and parse_rainfall_json() (copy from Week 5)
# 2. Load '../Data/scenarios/fungwong_202511.json'
# 3. Parse into GeoDataFrame
# 4. Filter to 花蓮縣 + 宜蘭縣
# 5. Remove stations with rain_1hr <= 0
# 6. Convert to EPSG:3826

def normalize_cwa_json(data):
    """Normalize CWA JSON data to flattened format"""
    normalized = []
    for record in data.get('records', {}).get('Station', []):
        geo_info = record.get('GeoInfo', {})
        rainfall = record.get('RainfallElement', {})
        coords = geo_info.get('Coordinates', [{}])[0]
        
        normalized_record = {
            'station_id': record.get('StationId'),
            'station_name': record.get('StationName'),
            'county': geo_info.get('CountyName'),
            'township': geo_info.get('TownName'),
            'lat': coords.get('StationLatitude'),
            'lon': coords.get('StationLongitude'),
            'rain_1hr': rainfall.get('Past1hr', {}).get('Precipitation'),
            'rain_3hr': rainfall.get('Past3hr', {}).get('Precipitation'),
            'rain_6hr': rainfall.get('Past6hr', {}).get('Precipitation'),
            'rain_12hr': rainfall.get('Past12hr', {}).get('Precipitation'),
            'rain_24hr': rainfall.get('Past24hr', {}).get('Precipitation'),
            'obs_time': record.get('ObsTime', {}).get('DateTime')
        }
        normalized.append(normalized_record)
    return normalized

def parse_rainfall_json(json_file_path):
    """Parse rainfall JSON file into GeoDataFrame"""
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    normalized_data = normalize_cwa_json(data)
    df = pd.DataFrame(normalized_data)
    
    # Convert data types
    df['lat'] = pd.to_numeric(df['lat'])
    df['lon'] = pd.to_numeric(df['lon'])
    df['rain_1hr'] = pd.to_numeric(df['rain_1hr'])
    
    # Create geometry
    geometry = [Point(lon, lat) for lon, lat in zip(df['lon'], df['lat'])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')
    
    return gdf

# Load and process data
try:
    # Load fungwong_202511.json from correct path
    rainfall_gdf = parse_rainfall_json('../Data/scenarios/fungwong_202511.json')
    print(f"Loaded {len(rainfall_gdf)} stations")
    
    # Filter to 花蓮縣 and 宜蘭縣
    study_counties = ['花蓮縣', '宜蘭縣']
    study_rain = rainfall_gdf[rainfall_gdf['county'].isin(study_counties)].copy()
    print(f"Stations in study area: {len(study_rain)}")
    
    # Remove stations with rain_1hr <= 0 (including -998)
    study_rain = study_rain[study_rain['rain_1hr'] > 0].copy()
    print(f"Stations with positive rainfall: {len(study_rain)}")
    
    # Convert to EPSG:3826
    study_rain_3826 = study_rain.to_crs('EPSG:3826')
    print(f"Converted to EPSG:3826: {study_rain_3826.crs}")
    
    # Extract coordinate arrays for Kriging / ML
    x = study_rain_3826.geometry.x.values  # Easting (meters)
    y = study_rain_3826.geometry.y.values  # Northing (meters)
    z = study_rain_3826['rain_1hr'].values

    print(f"Study area stations (rain > 0): {len(study_rain_3826)}")
    print(f"CRS: {study_rain_3826.crs}")
    print(f"\nTop 5 stations:")
    print(study_rain_3826.nlargest(5, 'rain_1hr')[['station_name', 'county', 'rain_1hr']].to_string(index=False))
    
except FileNotFoundError:
    print("Error: ../Data/scenarios/fungwong_202511.json not found")
    print("Please ensure the data file is in the correct location")
    # Create dummy data for testing
    dummy_data = {
        'station_name': ['Station_A', 'Station_B', 'Station_C', 'Station_D', 'Station_E'],
        'county': ['花蓮縣', '宜蘭縣', '花蓮縣', '宜蘭縣', '花蓮縣'],
        'rain_1hr': [10.5, 25.3, 15.2, 30.1, 8.7],
        'geometry': [Point(121.6 + i*0.01, 23.8 + i*0.01) for i in range(5)]
    }
    study_rain_3826 = gpd.GeoDataFrame(dummy_data, crs='EPSG:4326').to_crs('EPSG:3826')
    x = study_rain_3826.geometry.x.values
    y = study_rain_3826.geometry.y.values
    z = study_rain_3826['rain_1hr'].values

---

## Cell [2a]: Variogram — First Attempt (Naive)

Try running Kriging directly on the raw rainfall data. **Don't worry if it looks bad — that's the point.**

**Key Concepts** (will make sense after you see the result):
- **Nugget**: How noisy is each measurement?
- **Sill**: What's the maximum difference between stations?
- **Range**: How far apart do stations need to be before they stop "knowing about each other"?

**CRS Warning**: The x, y arrays MUST be in EPSG:3826 (meters). Using lat/lon will give wrong results.

In [ ]:
from pykrige.ok import OrdinaryKriging

# 🔴 First attempt: run Kriging on raw rainfall data
# YOUR CODE HERE:
# 1. Create OrdinaryKriging with raw z values
#    Use variogram_model='spherical', verbose=False, enable_plotting=True, nlags=15
# 2. Provide initial parameters to help the optimizer:
#    sill = z.var(), range = 50000, nugget = z.var() * 0.1

initial_sill = float(z.var())
initial_range = 50000.0
initial_nugget = float(z.var() * 0.1)

OK_naive = OrdinaryKriging(x, y, z, variogram_model='spherical',
                           verbose=False, enable_plotting=True, nlags=15,
                           variogram_parameters={'sill': initial_sill,
                                                 'range': initial_range,
                                                 'nugget': initial_nugget})

params = OK_naive.variogram_model_parameters
print(f"Sill:   {params[0]:.1f}")
print(f"Range:  {params[1]:.0f} m ({params[1]/1000:.1f} km)")
print(f"Nugget: {params[2]:.1f}")
print("\n⚠️ Look at the plot — do the dots follow the curve?")

## Cell [2b]: Why Did It Fail? — Look at the Histogram

The variogram looked bad. Before blaming the tool, **look at your data**.

Plot a histogram of the rainfall values. What do you see?

In [ ]:
# YOUR CODE HERE:
# 1. Plot a histogram of z (raw rainfall)
# 2. Plot a histogram of np.log1p(z) (log-transformed)
# 3. Compare: which one looks more "balanced"?

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(z, bins=30, color='tomato', edgecolor='black')
axes[0].set_title('Raw Rainfall (mm/hr)')
axes[0].set_xlabel('Rainfall (mm/hr)')

z_log = np.log1p(z)
axes[1].hist(z_log, bins=30, color='steelblue', edgecolor='black')
axes[1].set_title('After log(1+z)')
axes[1].set_xlabel('log(1 + Rainfall)')
plt.tight_layout()
plt.show()

print("Left: most stations < 10 mm, but a few are 50-130 mm.")
print("Those extreme values mess up the variogram.")
print("Right: after log-transform, the values are more balanced.")

## Cell [2c]: Variogram — Second Attempt (with Log-Transform)

Now redo the variogram using the log-transformed data. Compare this plot with Cell [2a] — it should be much better.

**Rule of thumb**: If your histogram has a long tail to the right, apply `np.log1p(z)` before Kriging, then use `np.expm1()` to convert back after prediction.

In [ ]:
# 🟢 Second attempt: Kriging on log-transformed data
# YOUR CODE HERE:
# 1. z_log = np.log1p(z)  (already computed above)
# 2. Create OrdinaryKriging with z_log (not z!)
# 3. Use initial parameters based on z_log.var()

initial_sill = float(z_log.var())
initial_range = 50000.0
initial_nugget = float(z_log.var() * 0.1)

OK = OrdinaryKriging(x, y, z_log, variogram_model='spherical',
                     verbose=False, enable_plotting=True, nlags=15,
                     variogram_parameters={'sill': initial_sill,
                                           'range': initial_range,
                                           'nugget': initial_nugget})

params = OK.variogram_model_parameters
print(f"Sill:   {params[0]:.3f}")
print(f"Range:  {params[1]:.0f} m ({params[1]/1000:.1f} km)")
print(f"Nugget: {params[2]:.3f}")
print("\n✅ Compare with Cell [2a] — the dots should follow the curve now.")

## Cell [2d]: Which Variogram Fits Best? — Range & Model Comparison (Slide 7)

Compare variograms **one variable at a time** — this is how scientists isolate effects:

- **Figure 1**: Fix model = **Spherical**, vary Range (50km, 25km, 15km) → what does Range do?
- **Figure 2**: Fix model = **Exponential**, vary Range (50km, 25km, 15km) → same exercise
- **Then compare**: same Range, different model → what does the model choice do?

**AI Prompt Suggestion**:
```
Create two sets of variogram plots using z_log data:
Figure 1: Spherical model with Range = 50km, 25km, 15km (1×3 subplots)
Figure 2: Exponential model with Range = 50km, 25km, 15km (1×3 subplots)
For each, plot empirical variogram (ok.lags vs ok.semivariance) as red dots
and the fitted model curve as a black line.
Compute SSE for each config and print a summary table.
Compare within each model first (Range effect), then across models (Model effect).
```

In [ ]:
# YOUR CODE HERE:
# 1. Define ranges: [50000, 25000, 15000]
# 2. Figure 1: Spherical × 3 Ranges (1×3 subplot)
#    For each: create OrdinaryKriging, plot lags vs semivariance (red dots),
#    plot fitted curve (black line), compute SSE
# 3. Figure 2: Exponential × 3 Ranges (1×3 subplot)
#    Same as above but with variogram_model='exponential'
# 4. Print summary table:
#    - Compare within Spherical (Range effect)
#    - Compare within Exponential (Range effect)
#    - Compare Spherical vs Exponential at same Range (Model effect)

from pykrige.ok import OrdinaryKriging

# Define ranges in meters
ranges_m = [50000, 25000, 15000]  # 50km, 25km, 15km
range_labels = ['50km', '25km', '15km']

# Function to compute SSE (Sum of Squared Errors)
def compute_sse(ok):
    """Compute SSE between empirical and fitted variogram"""
    empirical = ok.semivariance
    # Generate fitted values using the model parameters
    lags = ok.lags
    sill, range_param, nugget = ok.variogram_model_parameters
    
    if ok.variogram_model == 'spherical':
        # Spherical variogram function
        fitted = np.where(
            lags <= range_param,
            nugget + (sill - nugget) * (1.5 * lags / range_param - 0.5 * (lags / range_param) ** 3),
            sill
        )
    else:  # exponential
        # Exponential variogram function
        fitted = nugget + (sill - nugget) * (1 - np.exp(-3 * lags / range_param))
    
    return np.sum((empirical - fitted) ** 2)

# Store results for summary table
results = []

# ─── Figure 1: Spherical ───
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Spherical Variogram Model - Range Comparison', fontsize=14, fontweight='bold')

for ax, r_m, label in zip(axes, ranges_m, range_labels):
    # Create OrdinaryKriging with specified range
    ok_test = OrdinaryKriging(x, y, z_log, variogram_model='spherical',
                             verbose=False, enable_plotting=False, nlags=15,
                             variogram_parameters={'sill': float(z_log.var()),
                                                   'range': float(r_m),
                                                   'nugget': float(z_log.var() * 0.1)})
    
    # Plot empirical variogram (red dots)
    ax.scatter(ok_test.lags/1000, ok_test.semivariance, c='red', s=30, zorder=5, label='Empirical')
    
    # Plot fitted curve (black line)
    lags_smooth = np.linspace(0, max(ok_test.lags), 200)
    sill, range_param, nugget = ok_test.variogram_model_parameters
    fitted_smooth = np.where(
        lags_smooth <= range_param,
        nugget + (sill - nugget) * (1.5 * lags_smooth / range_param - 0.5 * (lags_smooth / range_param) ** 3),
        sill
    )
    ax.plot(lags_smooth/1000, fitted_smooth, 'k-', linewidth=2, label='Fitted')
    
    # Compute SSE
    sse = compute_sse(ok_test)
    results.append({
        'Model': 'Spherical',
        'Range': label,
        'SSE': sse,
        'Sill': sill,
        'Nugget': nugget
    })
    
    # Formatting
    ax.set_xlabel('Distance (km)')
    ax.set_ylabel('Semivariance')
    ax.set_title(f'Range = {label}')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()

# ─── Figure 2: Exponential ───
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle('Exponential Variogram Model - Range Comparison', fontsize=14, fontweight='bold')

for ax, r_m, label in zip(axes2, ranges_m, range_labels):
    # Create OrdinaryKriging with specified range
    ok_test = OrdinaryKriging(x, y, z_log, variogram_model='exponential',
                             verbose=False, enable_plotting=False, nlags=15,
                             variogram_parameters={'sill': float(z_log.var()),
                                                   'range': float(r_m),
                                                   'nugget': float(z_log.var() * 0.1)})
    
    # Plot empirical variogram (red dots)
    ax.scatter(ok_test.lags/1000, ok_test.semivariance, c='red', s=30, zorder=5, label='Empirical')
    
    # Plot fitted curve (black line)
    lags_smooth = np.linspace(0, max(ok_test.lags), 200)
    sill, range_param, nugget = ok_test.variogram_model_parameters
    fitted_smooth = nugget + (sill - nugget) * (1 - np.exp(-3 * lags_smooth / range_param))
    ax.plot(lags_smooth/1000, fitted_smooth, 'k-', linewidth=2, label='Fitted')
    
    # Compute SSE
    sse = compute_sse(ok_test)
    results.append({
        'Model': 'Exponential',
        'Range': label,
        'SSE': sse,
        'Sill': sill,
        'Nugget': nugget
    })
    
    # Formatting
    ax.set_xlabel('Distance (km)')
    ax.set_ylabel('Semivariance')
    ax.set_title(f'Range = {label}')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

# ─── Summary Table ───
results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("VARIogram MODEL COMPARISON SUMMARY")
print("="*80)
print(f"{'Model':<12} {'Range':<8} {'SSE':<12} {'Sill':<8} {'Nugget':<8}")
print("-" * 60)
for _, row in results_df.iterrows():
    print(f"{row['Model']:<12} {row['Range']:<8} {row['SSE']:<12.3f} {row['Sill']:<8.3f} {row['Nugget']:<8.3f}")

print("\n" + "="*80)
print("ANALYSIS:")
print("="*80)

# Within-model comparison (Range effect)
print("\n1. RANGE EFFECT (within each model):")
print("   Spherical:")
spherical_results = results_df[results_df['Model'] == 'Spherical'].sort_values('SSE')
for _, row in spherical_results.iterrows():
    print(f"     {row['Range']}: SSE = {row['SSE']:.3f}")
best_spherical = spherical_results.iloc[0]
print(f"     → Best range: {best_spherical['Range']} (SSE = {best_spherical['SSE']:.3f})")

print("\n   Exponential:")
exponential_results = results_df[results_df['Model'] == 'Exponential'].sort_values('SSE')
for _, row in exponential_results.iterrows():
    print(f"     {row['Range']}: SSE = {row['SSE']:.3f}")
best_exponential = exponential_results.iloc[0]
print(f"     → Best range: {best_exponential['Range']} (SSE = {best_exponential['SSE']:.3f})")

# Cross-model comparison (Model effect)
print("\n2. MODEL EFFECT (at same range):")
for range_label in range_labels:
    spherical_sse = results_df[(results_df['Model'] == 'Spherical') & (results_df['Range'] == range_label)]['SSE'].iloc[0]
    exponential_sse = results_df[(results_df['Model'] == 'Exponential') & (results_df['Range'] == range_label)]['SSE'].iloc[0]
    better_model = 'Spherical' if spherical_sse < exponential_sse else 'Exponential'
    print(f"   {range_label}: Spherical SSE = {spherical_sse:.3f}, Exponential SSE = {exponential_sse:.3f}")
    print(f"     → {better_model} performs better at {range_label}")

print("\n3. OVERALL BEST CONFIGURATION:")
overall_best = results_df.loc[results_df['SSE'].idxmin()]
print(f"   {overall_best['Model']} with {overall_best['Range']}: SSE = {overall_best['SSE']:.3f}")

print("\n💡 INSIGHTS:")
print("   • Lower SSE indicates better fit to empirical data")
print("   • Range parameter controls spatial correlation distance")
print("   • Model choice affects the shape of the variogram curve")

## Cell [3]: Define the Interpolation Grid & Run Kriging (Slide 8)

Create a regular grid covering the study area with 1000m resolution, then execute Kriging.

**Think about**:
- What units is the grid in? (Hint: same as your CRS = meters)
- Why add a buffer around the station extent?

**Important**: Kriging runs in log-space. After prediction, **back-transform** with `np.expm1()` to get rainfall in mm/hr.

In [ ]:
import time

# YOUR CODE HERE:
# 1. Calculate grid extent from x, y arrays (with 5km buffer)
# 2. Create grid_x and grid_y using np.arange with 1000m step
# 3. Execute Kriging in log-space: z_kriging_log, ss_kriging_log = OK.execute('grid', grid_x, grid_y)
# 4. Back-transform: z_kriging = np.expm1(z_kriging_log)

buffer_m = 5000
resolution = 1000  # meters — use 500 for finer resolution (slower)

x_min = x.min() - buffer_m
x_max = x.max() + buffer_m
y_min = y.min() - buffer_m
y_max = y.max() + buffer_m
grid_x = np.arange(x_min, x_max, resolution)
grid_y = np.arange(y_min, y_max, resolution)

print(f"Grid: {len(grid_x)}×{len(grid_y)} = {len(grid_x)*len(grid_y):,} points @ {resolution}m")

t0 = time.time()
z_kriging_log, ss_kriging_log = OK.execute('grid', grid_x, grid_y)
print(f"✓ Kriging (log-space) done in {time.time()-t0:.1f}s")

# Back-transform to real rainfall (mm/hr)
z_kriging = np.expm1(z_kriging_log)
z_kriging[z_kriging < 0] = 0
ss_kriging = ss_kriging_log  # keep log-space variance for Sigma Map

print(f"  z range (mm/hr): {np.nanmin(z_kriging):.1f} - {np.nanmax(z_kriging):.1f}")

---

## Cell [4]: Machine Learning — Random Forest Prediction (Slide 9)

**Captain's Log**: Treating coordinates as input features. ML learns `f(easting, northing) → rainfall`. Simple, but no uncertainty map.

**AI Prompt Suggestion**:
```
Train a RandomForestRegressor from scikit-learn using [easting, northing]
as features (X_train) and rain_1hr as target (y_train).
Use n_estimators=200, min_samples_leaf=3, random_state=42.
Then predict on the same grid as Kriging.
```

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# YOUR CODE HERE:
# 1. Prepare features: X_train = np.column_stack([x, y])
# 2. Train RandomForestRegressor
# 3. Create meshgrid from grid_x, grid_y
# 4. Predict on the grid

X_train = np.column_stack([x, y])
y_train = z

rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42)
rf.fit(X_train, y_train)
print(f"RF Training R²: {rf.score(X_train, y_train):.3f}")

grid_xx, grid_yy = np.meshgrid(grid_x, grid_y)
X_grid = np.column_stack([grid_xx.ravel(), grid_yy.ravel()])

t0 = time.time()
z_rf = rf.predict(X_grid).reshape(grid_xx.shape)
print(f"✓ Random Forest done in {time.time()-t0:.1f}s")
print(f"  z range: {z_rf.min():.1f} - {z_rf.max():.1f} mm/hr")

## Cell [5]: ML Glass Box — Feature Importance (Slide 11)

**Captain's Question**: "AI used what to predict floods — latitude or elevation?"

Even though ML is a "black box", we can peek inside with `.feature_importances_`.

In [ ]:
# YOUR CODE HERE:
# 1. Print rf.feature_importances_
# 2. Interpret: which dimension matters more for typhoon rainfall?

importances = rf.feature_importances_
print("Feature Importance:")
print(f"  Easting (X):  {importances[0]:.3f}")
print(f"  Northing (Y): {importances[1]:.3f}")
print(f"\nThe model relies mostly on {'easting' if importances[0] > importances[1] else 'northing'}.")
print("Think: does this make physical sense for Typhoon Fung-wong?")

---

## ⏸️ Lab 1: The Four-Way Interpolation Shootout (40 min)

### Cell [6]: Nearest Neighbor + IDW

Compute two additional interpolation methods so we can compare all four.

**AI Prompt Suggestion**:
```
Use scipy.interpolate.NearestNDInterpolator for nearest neighbor interpolation,
and manual IDW with scipy.spatial.distance.cdist (power=2) for IDW.
Both should predict on the same meshgrid as Kriging and RF.
Note: Don't use Rbf(function='inverse') — it's not real IDW and produces extreme values.
```

In [ ]:
from scipy.interpolate import NearestNDInterpolator
from scipy.spatial.distance import cdist

# YOUR CODE HERE:
# 1. Nearest Neighbor interpolation
# 2. IDW interpolation (手動實作，power=2)
#    ⚠️ 注意：不要用 Rbf(function='inverse')，它不是真正的 IDW，會產生極端值

nn_interp = NearestNDInterpolator(list(zip(x, y)), z)
z_nn = nn_interp(grid_xx, grid_yy)

pts = np.column_stack([x, y])
grid_pts = np.column_stack([grid_xx.ravel(), grid_yy.ravel()])
dists = cdist(grid_pts, pts)
dists[dists < 1] = 1  # 避免除以零
power = 2
weights = 1.0 / (dists ** power)
z_idw = ((weights @ z) / weights.sum(axis=1)).reshape(grid_xx.shape)

print("✓ Nearest Neighbor + IDW computed")

### Cell [7]: Four Methods Side by Side (Slide 13)

Create a 2×2 figure comparing all four interpolation methods.

**AI Prompt Suggestion**:
```
Create a 2×2 matplotlib figure comparing four interpolation results:
- Nearest Neighbor (z_nn)
- IDW (z_idw)
- Ordinary Kriging (z_kriging)
- Random Forest (z_rf)
Use YlOrRd colormap, same vmin/vmax, overlay station points in black.
Add descriptive subtitles (e.g., "Voronoi Patchwork", "Bullseye Effect",
"Smooth + Sigma Map", "ML Block Artifacts").
Save as 'interpolation_shootout.png'.
```

In [ ]:
# YOUR CODE HERE:
# 1. Create fig, axes = plt.subplots(2, 2, figsize=(18, 14))
# 2. Plot all four methods with imshow
# 3. Use extent=[x_min, x_max, y_min, y_max], origin='lower'
# 4. Overlay station scatter points
# 5. Add colorbars, titles, save figure

# Set up Chinese font for matplotlib
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
vmax = max(z) * 1.1
methods = [
     ('Nearest Neighbor\n(Voronoi / "Patchwork")', z_nn),
     ('IDW\n(Bullseye Effect)', z_idw),
     ('Ordinary Kriging\n(Smooth + Sigma Map)', z_kriging),
     ('Random Forest\n(ML "Block" Artifacts)', z_rf),
 ]

for ax, (title, data) in zip(axes.flatten(), methods):
     im = ax.imshow(data, extent=[x_min, x_max, y_min, y_max],
                    origin='lower', cmap='YlOrRd', vmin=0, vmax=vmax)
     ax.scatter(x, y, c='black', s=8, zorder=5)
     ax.set_title(title, fontsize=12, fontweight='bold')
     plt.colorbar(im, ax=ax, shrink=0.7, label='mm/hr')

plt.suptitle('Typhoon Fung-wong — Four Interpolation Methods Compared', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('interpolation_shootout.png', dpi=150, bbox_inches='tight')
plt.show()

### Cell [8]: Kriging vs RF — Direct Comparison

Create a 3-panel comparison: Kriging | Random Forest | Difference Map

The difference map reveals **where the two methods disagree** — these are the areas where the Commander needs extra caution.

In [ ]:
# YOUR CODE HERE:
# 1. Create fig, axes = plt.subplots(1, 3, figsize=(22, 7))
# 2. Left: Kriging (YlOrRd)
# 3. Middle: Random Forest (YlOrRd)
# 4. Right: Difference (Kriging - RF) using RdBu_r colormap
# 5. Save as 'kriging_vs_rf.png'

# Set up Chinese font for matplotlib
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Left: Kriging
im1 = axes[0].imshow(z_kriging, extent=[x_min, x_max, y_min, y_max],
                     origin='lower', cmap='YlOrRd', vmin=0, vmax=max(z) * 1.1)
axes[0].scatter(x, y, c='black', s=8, zorder=5)
axes[0].set_title('Ordinary Kriging', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Easting (m)')
axes[0].set_ylabel('Northing (m)')
plt.colorbar(im1, ax=axes[0], shrink=0.8, label='mm/hr')

# Middle: Random Forest
im2 = axes[1].imshow(z_rf, extent=[x_min, x_max, y_min, y_max],
                     origin='lower', cmap='YlOrRd', vmin=0, vmax=max(z) * 1.1)
axes[1].scatter(x, y, c='black', s=8, zorder=5)
axes[1].set_title('Random Forest', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Easting (m)')
axes[1].set_ylabel('Northing (m)')
plt.colorbar(im2, ax=axes[1], shrink=0.8, label='mm/hr')

# Right: Difference (Kriging - RF)
diff = z_kriging - z_rf
vmax_diff = max(abs(diff.min()), abs(diff.max()))
im3 = axes[2].imshow(diff, extent=[x_min, x_max, y_min, y_max],
                     origin='lower', cmap='RdBu_r', vmin=-vmax_diff, vmax=vmax_diff)
axes[2].scatter(x, y, c='black', s=8, zorder=5)
axes[2].set_title('Difference (Kriging - RF)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Easting (m)')
axes[2].set_ylabel('Northing (m)')
plt.colorbar(im3, ax=axes[2], shrink=0.8, label='mm/hr')

# Add overall title
plt.suptitle('Kriging vs Random Forest — Direct Comparison', fontsize=16, y=1.02)

# Save figure
plt.tight_layout()
plt.savefig('kriging_vs_rf.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary statistics
print(f"Kriging range: {z_kriging.min():.1f} - {z_kriging.max():.1f} mm/hr")
print(f"Random Forest range: {z_rf.min():.1f} - {z_rf.max():.1f} mm/hr")
print(f"Difference range: {diff.min():.1f} - {diff.max():.1f} mm/hr")
print(f"Mean absolute difference: {np.mean(np.abs(diff)):.2f} mm/hr")

### Lab 1 Reflection

#### 1. Which method produces the most physically realistic rainfall pattern? Why?

Kriging 產生最符合物理現實的降雨分布。因為 Kriging 考慮了空間自相關性（spatial autocorrelation），能夠反映距離與降雨之間的關係，因此結果通常較為平滑且連續。此外，其預測的高降雨區多半會圍繞實際測站分布，整體看起來較自然，不會出現明顯的人為痕跡。

相比之下：

- Nearest Neighbor 過於粗糙，呈現拼布狀（patchwork）
- IDW 會出現明顯的 bullseye（同心圓）現象
- Random Forest 容易出現區塊狀（block artifacts）

因此，Kriging 的結果最符合真實降雨的空間特性。

---

#### 2. Where do Kriging and RF disagree the most? What does this tell you?

Kriging 與 Random Forest 差異最大的位置，主要出現在高降雨區及其周圍邊界，也就是降雨變化最劇烈的地方。

Kriging 呈現的是較平滑、連續的變化；而 Random Forest 則常出現較明顯的區塊狀結構，預測值可能在某些地方突然改變。這表示 Kriging 能較好地捕捉空間連續性，但 Random Forest 在只使用 x、y 座標作為特徵時，較像是在對空間做切割，缺乏對真實空間結構的理解。

這也說明，如果要讓 RF 更符合現實，通常需要加入更多地形或環境特徵，例如高程、坡度等。

---

#### 3. What "artifacts" do you observe in the NN and RF results?

在 Nearest Neighbor（NN）中，可以觀察到以下 artifacts：

- 明顯的拼布狀效果（patchwork effect）
- 區域邊界過於銳利，不夠自然
- 缺乏空間連續性

在 Random Forest（RF）中，可以觀察到：

- 區塊狀 artifacts（block artifacts）
- 階梯狀變化（step-like transitions）
- 預測結果呈現不自然的空間切割

這些現象都屬於模型方法本身造成的人為痕跡，並不符合真實降雨通常應有的平滑變化特性。

---

#### 4. If you were the Commander, which method would you trust for evacuation decisions? Why?

如果我是指揮官，我會選擇相信 Kriging。

因為 Kriging 提供的降雨分布較平滑、自然，也較符合空間現象的物理特性。它不只考慮測站數值，也考慮測站之間的空間關係，因此預測結果通常比其他方法更可靠。此外，Kriging 還可以提供不確定性資訊（variance / uncertainty），這對於撤離決策非常重要，因為指揮官不只需要知道哪裡可能有高風險，也需要知道哪些地方的預測較不確定。

相比之下：

- NN 太粗糙
- IDW 容易出現不自然的集中現象
- RF 在只使用座標時容易出現區塊化問題

因此，Kriging 是最值得信任的撤離決策依據。

**Your Lab 1 reflection here:**

1. Most realistic: 直覺上Kriging表現依該要是最好的，因為他符合物理上的分布概念。距離越遠不確定性越高，同時也使用測站距離作為雨量高低的判斷基準，因此較NN更加優秀。
2. Disagreement areas: Kriging預測結果較為平滑，而ML預測結果較為方正，缺乏真實性。
3. Artifacts: 預測結果皆較為區塊化，和真實降雨應呈現的平滑分布有所不同。
4. Commander's choice: 我也會選擇Kriging，他是專門設計給這種空間資訊的內插做使用的。

---

## ⏸️ Lab 2: Confidence & Uncertainty Diagnosis (30 min)

### Cell [9]: The Sigma Map — Kriging's Unique Weapon (Slide 15)

**Captain's Log**: This is what makes Kriging special. No other method natively provides a confidence map.

- `ss_kriging` = Kriging variance at each grid point
- Low variance → many nearby stations → reliable estimate
- High variance → far from stations → uncertain

**For the Commander**:
- HIGH rainfall + LOW variance = CONFIRMED: Evacuate now
- HIGH rainfall + HIGH variance = UNCERTAIN: Deploy sensors first

In [ ]:
# YOUR CODE HERE:
# Create a 2-panel figure:
# Left: z_kriging with YlOrRd colormap (rainfall estimate)
# Right: ss_kriging with Blues colormap (variance/uncertainty)
# Add station locations on both panels (red dots on variance map)
# Save as 'sigma_map.png'

# Set up Chinese font for matplotlib
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Rainfall estimate
im1 = axes[0].imshow(z_kriging, extent=[x_min, x_max, y_min, y_max],
                       origin='lower', cmap='YlOrRd', vmin=0)
axes[0].scatter(x, y, c='black', s=10, zorder=5)
axes[0].set_title('Estimated Rainfall (mm/hr)')
plt.colorbar(im1, ax=axes[0], shrink=0.8, label='mm/hr')

# Kriging Variance (Sigma Map)
im2 = axes[1].imshow(ss_kriging, extent=[x_min, x_max, y_min, y_max],
                       origin='lower', cmap='Blues', vmin=0)
axes[1].scatter(x, y, c='red', s=10, zorder=5, label='Stations')
axes[1].set_title('Kriging Sigma Map (Uncertainty)')
axes[1].legend(loc='upper right')
plt.colorbar(im2, ax=axes[1], shrink=0.8, label='Variance')

plt.suptitle('The Honest Report Card: Estimate + Confidence', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('sigma_map.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Variance range: {np.nanmin(ss_kriging):.1f} - {np.nanmax(ss_kriging):.1f}")

### Cell [9b]: Nugget Effect — Why Extreme Rain Gets Diluted (Slide 7)

Suao recorded **130.5 mm/hr**, but default Kriging predicts only ~71 mm at 500m away. Why?

**Nugget** controls how much the model trusts the actual measurements:
- High Nugget = "measurements are noisy" → smooths everything → extreme values diluted
- Low Nugget = "measurements are accurate" → passes through stations → extremes preserved

**Task**: Compare Nugget=10% vs Nugget=1% on a zoomed-in map around Suao. Which preserves the extreme rainfall better?

**AI Prompt Suggestion**:
```
Create two OrdinaryKriging models on z_log with identical parameters except
nugget: one with nugget = sill * 0.10, one with nugget = sill * 0.01.
Predict on a 20km×20km grid centered on the station with maximum rainfall.
Show side-by-side maps and print predicted values at 0m, 500m, 1000m, 2000m
from that station. Which Nugget preserves the extreme value better?
```

In [ ]:
# YOUR CODE HERE:
# 1. Find the station with maximum rainfall (suao_idx = np.argmax(z))
# 2. Create two OrdinaryKriging models: nugget = sill*0.10 and sill*0.01
# 3. Predict on a local grid (20km box) around that station
# 4. Plot side-by-side comparison maps
# 5. Predict at specific offsets: 0m, 500m, 1000m, 2000m from the station
# 6. Print comparison table

# Set up Chinese font for matplotlib
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# Find station with maximum rainfall
suao_idx = np.argmax(z)
suao_x, suao_y, suao_z = x[suao_idx], y[suao_idx], z[suao_idx]
suao_name = study_rain_3826.iloc[suao_idx]['station_name']
print(f"Station with max rainfall: {suao_name} - {suao_z:.1f} mm/hr")
print(f"Location: ({suao_x:.0f}, {suao_y:.0f})")

# Create two Kriging models with different nugget values
sill_val = float(z_log.var())
range_val = 25000.0  # Use best range from previous analysis (25km)

print(f"\nCreating two Kriging models:")
print(f"  Sill: {sill_val:.3f}")
print(f"  Range: {range_val:.0f} m")
print(f"  Model 1 - Nugget: {sill_val * 0.10:.3f} (10% of sill)")
print(f"  Model 2 - Nugget: {sill_val * 0.01:.3f} (1% of sill)")

# Model 1: High nugget (10% of sill)
OK_high_nugget = OrdinaryKriging(x, y, z_log, variogram_model='spherical',
                                 verbose=False, enable_plotting=False,
                                 variogram_parameters={'sill': sill_val,
                                                       'range': range_val,
                                                       'nugget': sill_val * 0.10})

# Model 2: Low nugget (1% of sill)
OK_low_nugget = OrdinaryKriging(x, y, z_log, variogram_model='spherical',
                                verbose=False, enable_plotting=False,
                                variogram_parameters={'sill': sill_val,
                                                      'range': range_val,
                                                      'nugget': sill_val * 0.01})

# Create local grid (20km × 20km box around Suao)
local_extent = 10000  # 10km in each direction = 20km × 20km box
local_resolution = 100  # 100m resolution for detailed analysis

local_x_min = suao_x - local_extent
local_x_max = suao_x + local_extent
local_y_min = suao_y - local_extent
local_y_max = suao_y + local_extent

local_grid_x = np.arange(local_x_min, local_x_max, local_resolution)
local_grid_y = np.arange(local_y_min, local_y_max, local_resolution)

print(f"\nLocal grid: {len(local_grid_x)}×{len(local_grid_y)} = {len(local_grid_x)*len(local_grid_y):,} points @ {local_resolution}m")

# Predict with both models
z_high_log, _ = OK_high_nugget.execute('grid', local_grid_x, local_grid_y)
z_low_log, _ = OK_low_nugget.execute('grid', local_grid_x, local_grid_y)

# Back-transform to rainfall values
z_high = np.expm1(z_high_log)
z_low = np.expm1(z_low_log)

# Predict at specific distances from the station
distances = [0, 500, 1000, 2000]  # meters
print(f"\nPredicted rainfall at different distances from {suao_name}:")
print(f"{'Distance':<10} {'High Nugget':<12} {'Low Nugget':<11} {'Observed':<10}")
print("-" * 45)

for dist in distances:
    # Predict at point (suao_x + dist, suao_y)
    pred_high_log, _ = OK_high_nugget.execute('points', [suao_x + dist], [suao_y])
    pred_low_log, _ = OK_low_nugget.execute('points', [suao_x + dist], [suao_y])
    
    pred_high = np.expm1(pred_high_log[0])
    pred_low = np.expm1(pred_low_log[0])
    
    observed = suao_z if dist == 0 else "N/A"
    
    print(f"{dist:<10}m {pred_high:<12.1f} {pred_low:<11.1f} {observed:<10}")

# Create side-by-side comparison plots
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# High nugget plot
im1 = axes[0].imshow(z_high, extent=[local_x_min, local_x_max, local_y_min, local_y_max],
                     origin='lower', cmap='YlOrRd', vmin=0, vmax=suao_z*1.1)
axes[0].scatter(x, y, c='black', s=15, zorder=5, alpha=0.6)
axes[0].scatter(suao_x, suao_y, c='red', s=100, zorder=6, marker='*', 
                label=f'{suao_name} ({suao_z:.1f} mm/hr)')
axes[0].set_title(f'High Nugget (10% of sill)\nExtreme values get SMOOTHED', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Easting (m)')
axes[0].set_ylabel('Northing (m)')
axes[0].legend(loc='upper right')
plt.colorbar(im1, ax=axes[0], shrink=0.8, label='mm/hr')

# Low nugget plot
im2 = axes[1].imshow(z_low, extent=[local_x_min, local_x_max, local_y_min, local_y_max],
                     origin='lower', cmap='YlOrRd', vmin=0, vmax=suao_z*1.1)
axes[1].scatter(x, y, c='black', s=15, zorder=5, alpha=0.6)
axes[1].scatter(suao_x, suao_y, c='red', s=100, zorder=6, marker='*',
                label=f'{suao_name} ({suao_z:.1f} mm/hr)')
axes[1].set_title(f'Low Nugget (1% of sill)\nExtreme values PRESERVED', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Easting (m)')
axes[1].set_ylabel('Northing (m)')
axes[1].legend(loc='upper right')
plt.colorbar(im2, ax=axes[1], shrink=0.8, label='mm/hr')

plt.suptitle(f'Nugget Effect Comparison: {suao_name} Station\nHow Nugget Value Affects Extreme Rainfall Preservation', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('nugget_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Analysis summary
print(f"\n" + "="*60)
print("NUGGET EFFECT ANALYSIS:")
print("="*60)
print(f"📍 Station: {suao_name}")
print(f"🌧️  Observed rainfall: {suao_z:.1f} mm/hr")
print(f"📏 Distance 500m from station:")
print(f"   High nugget (10%): {np.expm1(OK_high_nugget.execute('points', [suao_x + 500], [suao_y])[0][0]):.1f} mm/hr")
print(f"   Low nugget (1%):  {np.expm1(OK_low_nugget.execute('points', [suao_x + 500], [suao_y])[0][0]):.1f} mm/hr")
print(f"   Difference: {np.expm1(OK_low_nugget.execute('points', [suao_x + 500], [suao_y])[0][0]) - np.expm1(OK_high_nugget.execute('points', [suao_x + 500], [suao_y])[0][0]):.1f} mm/hr")

print(f"\n🔑 Which Nugget preserves extreme values better?")
print("   → LOW NUGGET (1% of sill) preserves extreme rainfall better")
print("   → High nugget smooths the data, reducing extreme values")
print("   → Low nugget trusts the measurements more, passing through stations")

print(f"\n💡 Recommendation for CWA calibrated stations:")
print("   Use LOW NUGGET (1-5% of sill) for:")
print("   • Extreme rainfall events (typhoons, thunderstorms)")
print("   • Well-calibrated, high-quality stations")
print("   • When preserving extreme values is critical for warnings")
print("   • Use HIGH NUGGET only when data quality is questionable")

# 🔑 Which Nugget setting is better for CWA calibrated stations? Why?
print(f"\n🎯 ANSWER: Low nugget (1% of sill) is better for CWA calibrated stations")
print(f"   Reason: CWA stations are professionally maintained and calibrated,")
print(f"   so we should trust the measurements. Low nugget preserves the")
print(f"   extreme rainfall values that are critical for flood warnings.")

### Cell [10]: Export to GeoTIFF (Slide 16)

⚠️ **Flip warning**: `z_kriging` row 0 = south (numpy convention). GeoTIFF row 0 = north. Use `np.flipud()` to fix.

**AI Prompt Suggestion**:
```
Save 2D numpy arrays as GeoTIFF using rasterio. I need:
- from_bounds transform with my grid extent (x_min, y_min, x_max, y_max)
- CRS = EPSG:3826, dtype = float32
- Apply np.flipud() before writing
- Save kriging_rainfall.tif, kriging_variance.tif, and rf_rainfall.tif
```

In [ ]:
import rasterio
from rasterio.transform import from_bounds

# YOUR CODE HERE:
# 1. Compute rasterio transform using from_bounds
# 2. Write a helper function save_geotiff(data, filename)
# 3. Save kriging_rainfall.tif, kriging_variance.tif, and rf_rainfall.tif
# Remember: np.flipud() before writing!

def save_geotiff(data, filename, crs='EPSG:3826', nodata=-9999):
    """
    Save 2D numpy array as GeoTIFF file
    
    Parameters:
    -----------
    data : 2D numpy array
        The data to save
    filename : str
        Output filename
    crs : str, optional
        Coordinate reference system (default: 'EPSG:3826')
    nodata : float, optional
        NoData value (default: -9999)
    """
    # Apply flipud to fix row orientation (numpy row 0 = south, GeoTIFF row 0 = north)
    data_flipped = np.flipud(data).astype(np.float32)
    
    # Create rasterio transform
    transform = from_bounds(x_min, y_min, x_max, y_max,
                           width=data_flipped.shape[1], 
                           height=data_flipped.shape[0])
    
    # Write GeoTIFF
    with rasterio.open(filename, 'w', driver='GTiff',
                      height=data_flipped.shape[0], 
                      width=data_flipped.shape[1],
                      count=1, dtype='float32', 
                      crs=crs, transform=transform, 
                      nodata=nodata) as dst:
        dst.write(data_flipped, 1)
    
    print(f"✓ Saved {filename}")
    print(f"  Shape: {data_flipped.shape}")
    print(f"  CRS: {crs}")
    print(f"  Transform: {transform}")
    print(f"  Data range: {np.nanmin(data):.3f} - {np.nanmax(data):.3f}")
    print(f"  NoData: {nodata}")

print("Saving GeoTIFF files...")
print("=" * 50)

# Save Kriging rainfall estimate
save_geotiff(z_kriging, 'kriging_rainfall.tif')

print()

# Save Kriging variance (uncertainty)
save_geotiff(ss_kriging, 'kriging_variance.tif')

print()

# Save Random Forest rainfall estimate
save_geotiff(z_rf, 'rf_rainfall.tif')

print("=" * 50)
print("All GeoTIFF files saved successfully!")
print("\nFiles created:")
print("  • kriging_rainfall.tif - Ordinary Kriging rainfall estimate")
print("  • kriging_variance.tif - Kriging uncertainty (sigma map)")
print("  • rf_rainfall.tif - Random Forest rainfall estimate")
print("\nYou can open these files in QGIS, ArcGIS, or any GIS software to verify:")
print("  • Coordinate system: EPSG:3826 (TWD97)")
print("  • Extent matches your study area with 5km buffer")
print("  • Pixel resolution: 1000m × 1000m")

### Cell [11]: Zonal Statistics — Township Decision Table

Compute per-township statistics from your Kriging and RF rasters, then compare them side-by-side.

**Required output**: A DataFrame with: 鄉鎮 | Kriging平均 | Kriging最大 | RF平均 | 平均variance | 可信度

In [ ]:
from rasterstats import zonal_stats

# YOUR CODE HERE:
# 1. Load township boundaries (TGOS shapefile)
# 2. Filter to 花蓮縣 + 宜蘭縣, convert to EPSG:3826
# 3. Run zonal_stats on kriging_rainfall.tif, kriging_variance.tif, rf_rainfall.tif
# 4. Create summary DataFrame with columns:
#    鄉鎮, 縣市, Kriging平均, Kriging最大, RF平均, 平均variance
# 5. Add 可信度 column:
#    HIGH: variance < 33rd percentile
#    MEDIUM: 33rd-66th percentile
#    LOW: > 66th percentile

# Debug: Check if GeoTIFF files exist first
import os
print("Checking required files...")
required_files = ['kriging_rainfall.tif', 'kriging_variance.tif', 'rf_rainfall.tif']
for file in required_files:
    if os.path.exists(file):
        print(f"✓ {file} exists")
    else:
        print(f"❌ {file} missing - please run Cell [31] first")

try:
    # Load township boundaries
    print("\nLoading township boundaries...")
    shapefile_path = '../Data/TOWN_MOI_1140318/TOWN_MOI_1140318.shp'
    
    if not os.path.exists(shapefile_path):
        raise FileNotFoundError(f"Shapefile not found: {shapefile_path}")
    
    towns = gpd.read_file(shapefile_path)
    print(f"✓ Loaded shapefile with {len(towns)} townships")
    print(f"Available columns: {list(towns.columns)}")
    
    # Check county names in the shapefile
    if 'COUNTYNAME' in towns.columns:
        unique_counties = towns['COUNTYNAME'].unique()
        print(f"Available counties: {unique_counties}")
    else:
        print("Available columns for county names:")
        for col in towns.columns:
            if 'COUNTY' in col.upper() or 'COUNT' in col.upper():
                print(f"  - {col}")
    
    # Filter to study area (花蓮縣 + 宜蘭縣)
    study_counties = ['花蓮縣', '宜蘭縣']
    
    # Try different possible county column names
    county_col = None
    for col in ['COUNTYNAME', 'COUNTY_NAME', 'COUNTY', 'COUNTYNM']:
        if col in towns.columns:
            county_col = col
            break
    
    if county_col is None:
        raise ValueError("Could not find county name column in shapefile")
    
    print(f"Using county column: {county_col}")
    study_towns = towns[towns[county_col].isin(study_counties)].copy()
    print(f"Townships in study area: {len(study_towns)}")
    
    if len(study_towns) == 0:
        print("No townships found. Available counties:")
        print(towns[county_col].value_counts())
        raise ValueError("No matching townships found for study counties")
    
    # Convert to EPSG:3826 (same as our rainfall data)
    study_towns = study_towns.to_crs(epsg=3826)
    print(f"✓ Converted to EPSG:3826: {study_towns.crs}")
    
    # Check township name column
    town_col = None
    for col in ['TOWNNAME', 'TOWN_NAME', 'TOWN', 'TOWNNM']:
        if col in study_towns.columns:
            town_col = col
            break
    
    if town_col is None:
        raise ValueError("Could not find township name column in shapefile")
    
    print(f"Using township column: {town_col}")
    
    print("\nComputing zonal statistics...")
    
    # Compute zonal statistics for each raster
    # Kriging rainfall statistics
    print("Computing Kriging rainfall stats...")
    kriging_stats = zonal_stats(
        study_towns, 
        'kriging_rainfall.tif',
        stats=['mean', 'max', 'count'],
        geojson_out=False
    )
    print(f"✓ Kriging stats computed: {len(kriging_stats)} results")
    
    # Kriging variance statistics
    print("Computing Kriging variance stats...")
    variance_stats = zonal_stats(
        study_towns,
        'kriging_variance.tif', 
        stats=['mean'],
        geojson_out=False
    )
    print(f"✓ Variance stats computed: {len(variance_stats)} results")
    
    # Random Forest rainfall statistics
    print("Computing Random Forest stats...")
    rf_stats = zonal_stats(
        study_towns,
        'rf_rainfall.tif',
        stats=['mean', 'count'],
        geojson_out=False
    )
    print(f"✓ RF stats computed: {len(rf_stats)} results")
    
    # Verify all stats have same length
    if not (len(kriging_stats) == len(variance_stats) == len(rf_stats) == len(study_towns)):
        print(f"❌ Length mismatch:")
        print(f"  study_towns: {len(study_towns)}")
        print(f"  kriging_stats: {len(kriging_stats)}")
        print(f"  variance_stats: {len(variance_stats)}")
        print(f"  rf_stats: {len(rf_stats)}")
        raise ValueError("Zonal stats length mismatch")
    
    print("✓ All zonal statistics computed and verified")
    
    # Create summary DataFrame - FIXED VERSION
    summary_data = []
    
    for idx, (_, town) in enumerate(study_towns.iterrows()):
        town_name = town[town_col]
        county_name = town[county_col]
        
        # Safety check for index bounds
        if idx >= len(kriging_stats) or idx >= len(variance_stats) or idx >= len(rf_stats):
            print(f"❌ Index {idx} out of range for stats arrays")
            continue
        
        # Extract statistics with safety checks
        kriging_stat = kriging_stats[idx] if idx < len(kriging_stats) else {}
        variance_stat = variance_stats[idx] if idx < len(variance_stats) else {}
        rf_stat = rf_stats[idx] if idx < len(rf_stats) else {}
        
        kriging_mean = kriging_stat.get('mean', 0) if kriging_stat.get('mean') is not None else 0
        kriging_max = kriging_stat.get('max', 0) if kriging_stat.get('max') is not None else 0
        variance_mean = variance_stat.get('mean', 0) if variance_stat.get('mean') is not None else 0
        rf_mean = rf_stat.get('mean', 0) if rf_stat.get('mean') is not None else 0
        
        summary_data.append({
            '鄉鎮': town_name,
            '縣市': county_name,
            'Kriging平均': round(kriging_mean, 1),
            'Kriging最大': round(kriging_max, 1),
            'RF平均': round(rf_mean, 1),
            '平均variance': round(variance_mean, 3)
        })
    
    # Create DataFrame
    summary_df = pd.DataFrame(summary_data)
    
    if len(summary_df) == 0:
        raise ValueError("No valid data collected for summary")
    
    # Add confidence level based on variance percentiles
    variance_33 = np.percentile(summary_df['平均variance'], 33)
    variance_66 = np.percentile(summary_df['平均variance'], 66)
    
    def get_confidence(variance):
        if variance < variance_33:
            return 'HIGH'
        elif variance < variance_66:
            return 'MEDIUM'
        else:
            return 'LOW'
    
    summary_df['可信度'] = summary_df['平均variance'].apply(get_confidence)
    
    # Sort by county and township
    summary_df = summary_df.sort_values(['縣市', '鄉鎮']).reset_index(drop=True)
    
    # Display results
    print("\n" + "="*80)
    print("TOWNSHIP RAINFALL SUMMARY TABLE")
    print("="*80)
    print(summary_df.to_string(index=False))
    
    print(f"\nConfidence thresholds:")
    print(f"  HIGH: variance < {variance_33:.3f}")
    print(f"  MEDIUM: {variance_33:.3f} ≤ variance < {variance_66:.3f}")
    print(f"  LOW: variance ≥ {variance_66:.3f}")
    
    # Additional analysis
    print(f"\nSummary Statistics:")
    print(f"  Total townships: {len(summary_df)}")
    print(f"  HIGH confidence: {len(summary_df[summary_df['可信度'] == 'HIGH'])}")
    print(f"  MEDIUM confidence: {len(summary_df[summary_df['可信度'] == 'MEDIUM'])}")
    print(f"  LOW confidence: {len(summary_df[summary_df['可信度'] == 'LOW'])}")
    
    # Find extreme values
    if len(summary_df) > 0:
        max_kriging = summary_df.loc[summary_df['Kriging最大'].idxmax()]
        max_rf = summary_df.loc[summary_df['RF平均'].idxmax()]
        
        print(f"\nExtreme rainfall areas:")
        print(f"  Max Kriging: {max_kriging['鄉鎮']} ({max_kriging['縣市']}) - {max_kriging['Kriging最大']} mm/hr")
        print(f"  Max RF: {max_rf['鄉鎮']} ({max_rf['縣市']}) - {max_rf['RF平均']} mm/hr")
    
    # Save summary table
    summary_df.to_csv('township_rainfall_summary.csv', index=False, encoding='utf-8-sig')
    print(f"\n✓ Summary table saved as 'township_rainfall_summary.csv'")
    
except FileNotFoundError as e:
    print(f"❌ File not found: {e}")
    print("\nPlease check:")
    print("  1. GeoTIFF files exist (run Cell [31] first)")
    print("  2. Township shapefile path is correct")
    
except ValueError as e:
    print(f"❌ Data error: {e}")
    
except Exception as e:
    print(f"❌ Unexpected error: {e}")
    print(f"Error type: {type(e).__name__}")
    import traceback
    print("Full error details:")
    print(traceback.format_exc())

### Cell [12]: Thinking Challenge — Why Can't ML Give You a Sigma Map?

#### 1. Why does Kriging **natively** produce a variance map, but Random Forest does not?

Kriging 是基於空間統計模型的方法，特別是依賴變異數函數（variogram）來描述空間自相關性。在進行預測時，Kriging 不只估計某個位置的數值，同時也會計算該預測的不確定性（variance），因此可以自然產生一張 Sigma Map（變異數地圖）。

相對地，Random Forest 屬於機器學習方法，主要透過多棵決策樹學習輸入與輸出之間的關係。它的設計目標是提升預測準確度，而不是描述空間結構或不確定性，因此不會「原生」提供 variance map。

---

#### 2. Could you approximate uncertainty from RF using bootstrap or tree variance? What are the limitations?

可以透過一些方法近似 Random Forest 的不確定性，例如：

- 利用不同樹（trees）之間的預測變異（tree variance）
- 使用 bootstrap 重抽樣多次訓練模型，觀察預測分布

然而這些方法有幾個限制：

1. 這些不確定性並不具有明確的空間統計意義（不像 Kriging 的 variance）
2. 無法反映空間距離或測站分布對不確定性的影響
3. 不同方法得到的結果可能不一致，缺乏理論基礎支撐
4. 計算成本較高，且解釋性較弱

因此，RF 的不確定性通常只是「模型內部變異」，而不是「空間預測信心」。

---

#### 3. In your zonal stats table, which townships show **HIGH rainfall + LOW confidence**? What should the Commander do about those?

在 zonal stats 表中，「HIGH rainfall + LOW confidence」通常指：

- Kriging平均或最大雨量偏高  
- 但平均 variance 也偏高（被分類為 LOW）

這類鄉鎮代表：
👉 雖然預測顯示降雨量高，但模型對該區的信心不足，通常是因為該區距離測站較遠或測站分布稀疏。

對於這些區域，指揮官應採取較保守且謹慎的策略，例如：

1. 提高警戒等級，避免低估風險  
2. 優先部署更多即時監測設備（如雨量站）  
3. 持續更新預測模型與觀測資料  
4. 必要時提前進行預防性撤離  

總體而言，這些區域屬於「高風險但低確定性」，在防災決策中應特別關注。

**Your Lab 2 reflection here:**

1. Kriging vs ML uncertainty: Kriging式空間統計模型，因此可根據真實資料給出不去定性。而regression model僅根據座標當作特徵並以降雨量作為預測值，無法給出不確定性。
2. ML uncertainty approximation: 可用不同分類樹之間的變異數去做表示。但是這比較像是模型自己內部的變異數，和真實的資料蝶結較低。不過這不代表模型不可用，若有大量數據可進行訓練是有可能提升模型預測精準度的。
3. High rain + low confidence townships: 代表可能有強降雨，但較不確定。在該情況下指揮官應該要加強監測隨時採取必要措施。

### Cell [13]: (Bonus) AI Advisor Consultation

Ask an AI model (Gemini, ChatGPT, or Claude):

> 「在花蓮山區，測站密度約 1 站 / 50 km²。我用 Kriging 和 Random Forest 分別做了降雨內插，結果在山區差異很大。Kriging 的 variance 在山區也很高。請問：(1) 我應該信哪個結果？(2) 如何改善山區的預測品質？」

Paste the AI's response below and write your own commentary.

**AI Response:**

在花蓮山區這種測站稀疏（約 1 站 / 50 km²）且地形複雜的環境下，Kriging 與 Random Forest 出現明顯差異，且 Kriging variance 偏高，代表該區屬於「高不確定性區域」，兩種方法的預測結果都需要審慎解讀。

---

### (1) 我應該信哪個結果？

在這種情況下，不應該單純選擇相信某一個模型，而應該這樣理解：

- **Kriging 的優勢在於提供不確定性（variance）**
  - variance 偏高代表模型對該區預測信心不足
  - 因此 Kriging 至少能「誠實反映不可靠區域」

- **Random Forest 的限制**
  - 若只使用 (x, y) 作為輸入，無法反映山區地形影響
  - 容易產生不具物理意義的空間分割（block artifacts）

👉 因此：
- 若 RF 未納入地形變數，應較保守地參考 Kriging  
- 若 RF 已納入高程、坡度等變數，可作為補充，但仍需搭配 Kriging variance 判讀  

👉 重點不是選哪一個，而是：
**山區結果本身「不可靠」，需要一起看「數值 + 不確定性」**

---

### (2) 如何改善山區的預測品質？

#### 1. 增加觀測資料（最重要）

- 增設雨量站
- 整合雷達降雨資料
- 使用衛星降雨產品
- 提升測站密度

👉 測站稀疏是造成不確定性的主要原因

---

#### 2. 納入地形與環境因子

山區降雨強烈受地形控制，建議加入：

- elevation（高程）
- slope（坡度）
- aspect（坡向）
- 地形粗糙度
- 與稜線／谷地的距離
- 迎風／背風條件

👉 對 Random Forest 特別重要

---

#### 3. 採用混合方法（進階）

- Regression Kriging
- Co-kriging
- RF + Kriging（先用 RF，再對殘差做 Kriging）

👉 結合「地形解釋能力」與「空間統計能力」

---

### (3) 決策建議（Commander 視角）

對於「高降雨 + 高 variance」的山區：

- 視為 **高風險但低確定性區域**
- 採取較保守策略：
  - 提高警戒等級
  - 加強即時監測
  - 必要時提前預警或撤離

---

### 總結

在山區測站稀疏情況下：

- Kriging 提供「不確定性資訊」
- RF 提供「潛在模式（需搭配地形變數）」

👉 最重要的是：
**不要只看預測值，而要同時考慮預測的不確定性**

**My Commentary:**

我同意AI說的，我們要相信的是Kriginig的變異數資訊而非預測本身，所以季ˋ然variance很高的話那就代表資料可信度較低，而ML的產師結果本身可信度就不高，因此除非有納入更多的特徵否則可信度也是不高的。

---

## ARIA Evolution Recap

| Version | Week | Capability | Key Question |
|---------|------|-----------|---------------|
| v1.0 | W3 | River buffer + shelters | How close to the river? |
| v2.0 | W4 | + DEM terrain analysis | How steep is the terrain? |
| v3.0 | W5 | + Real-time rainfall stations | How much rain RIGHT NOW? |
| **v3.5** | **W6** | **+ Kriging & ML interpolation** | **What about areas with NO station? How confident are we?** |
| v4.0 | Final Project | Your extension! | Your question! |

---

*"Interpolation is not just filling space; it is predicting risk where sensors cannot reach."*